# Eje 1d — Paneles de opciones con Python

Derivados Financieros — QUANt UCEMA

Puente hacia el **Eje 4** (Market Data). Mismos módulos que usa la webapp UCEMA QUANT.

**Plan del notebook:**
1. Objetivo: paneles limpios
2. Spot y cotización (`Codigo.data.market_data`)
3. Vencimientos y options chain
4. Lectura rápida del panel (ATM, mid)
5. Nota sobre BYMA / datos frágiles
6. Resumen


## 1) Objetivo: paneles limpios

En clase y en la app necesitamos:
- **Spot** confiable
- Lista de **expiries**
- **Cadena** con bid/ask/last/IV

La webapp (*Market Data*) usa `Codigo.data.market_data` (con fallbacks). **No** uses scrapers frágiles como demo principal.


## 2) Spot y cotización

Si la red falla, el notebook no debe romperse: capturamos el error y seguimos con un mensaje claro.


In [1]:
import sys
sys.path.append('../../..')

from Codigo.data.market_data import get_spot, get_quote, get_expirations, get_options_chain

TICKER = 'AAPL'

try:
    spot = get_spot(TICKER)
    quote = get_quote(TICKER)
    print(f'{TICKER} spot = {spot:.2f}')
    print('quote keys:', sorted(quote.keys()) if isinstance(quote, dict) else type(quote))
    if isinstance(quote, dict):
        for k in ('price', 'change', 'change_pct', 'source', 'currency'):
            if k in quote:
                print(f'  {k}: {quote[k]}')
except Exception as e:
    spot = None
    print('No se pudo obtener spot/quote (red / proveedor).')
    print('Error:', type(e).__name__, e)
    print('→ En clase: reintentar, cambiar ticker, o continuar con el Eje 4 offline.')


AAPL spot = 336.13
quote keys: ['change', 'changePercent', 'last', 'name', 'source']
  change: -0.869995
  source: yahooquery


## 3) Vencimientos y options chain

Elegimos el primer vencimiento disponible y pedimos la cadena.


In [2]:
try:
    exps = get_expirations(TICKER)
    print(f'{len(exps)} vencimientos. Primeros 5: {exps[:5]}')
    expiry = exps[0]
    chain = get_options_chain(TICKER, expiry)
    print(f'Cadena {expiry}: {len(chain)} filas, columnas={list(chain.columns)}')
    display(chain.head(8))
except Exception as e:
    exps, expiry, chain = [], None, None
    print('No se pudo cargar la cadena.')
    print('Error:', type(e).__name__, e)


20 vencimientos. Primeros 5: ['2026-09-21', '2026-09-23', '2026-09-25', '2026-10-02', '2026-10-09']


Cadena 2026-09-21: 66 filas, columnas=['symbol', 'expiration', 'type', 'contractSymbol', 'strike', 'currency', 'lastPrice', 'change', 'percentChange', 'volume', 'openInterest', 'bid', 'ask', 'contractSize', 'lastTradeDate', 'impliedVolatility', 'inTheMoney', 'Spot', 'Ticker']


,symbol,expiration,type,contractSymbol,strike,currency,lastPrice,change,percentChange,volume,openInterest,bid,ask,contractSize,lastTradeDate,impliedVolatility,inTheMoney,Spot,Ticker
0,AAPL,2026-09-21,call,AAPL260921C00260000,260.0,USD,74.90,-0.979996,-1.291507,2.0,45.0,74.80,77.65,REGULAR,2026-09-18 16:50:24,1.195317,True,336.130005,AAPL
1,AAPL,2026-09-21,call,AAPL260921C00270000,270.0,USD,64.99,0.000000,0.000000,2.0,2.0,64.80,67.65,REGULAR,2026-09-17 16:15:47,1.037114,True,336.130005,AAPL
2,AAPL,2026-09-21,call,AAPL260921C00280000,280.0,USD,56.60,2.759998,5.126297,2.0,3.0,54.80,57.35,REGULAR,2026-09-18 19:52:15,1.377445,True,336.130005,AAPL
3,AAPL,2026-09-21,call,AAPL260921C00285000,285.0,USD,50.34,0.000000,0.000000,1.0,2.0,49.80,52.65,REGULAR,2026-09-17 13:30:03,0.806643,True,336.130005,AAPL
4,AAPL,2026-09-21,call,AAPL260921C00290000,290.0,USD,42.80,1.570000,3.807906,20.0,44.0,44.95,47.65,REGULAR,2026-09-18 14:44:27,0.796877,True,336.130005,AAPL
5,AAPL,2026-09-21,call,AAPL260921C00292500,292.5,USD,42.67,14.599998,52.012817,1.0,1.0,42.30,45.15,REGULAR,2026-09-18 13:59:20,0.695316,True,336.130005,AAPL
6,AAPL,2026-09-21,call,AAPL260921C00295000,295.0,USD,40.88,0.000000,0.000000,54.0,14.0,39.80,42.65,REGULAR,2026-09-17 17:37:38,0.658207,True,336.130005,AAPL
7,AAPL,2026-09-21,call,AAPL260921C00297500,297.5,USD,34.64,0.000000,0.000000,0.0,1.0,37.30,40.15,REGULAR,2026-09-11 14:08:53,0.621098,True,336.130005,AAPL


## 4) Lectura rápida — ATM y mid

- **Mid** ≈ (bid+ask)/2 cuando hay mercado
- **ATM** ≈ strike más cercano al spot
- En la app: página *Market Data* hace esto con filtros y smile de IV


In [3]:
if chain is not None and spot is not None and len(chain):
    df = chain.copy()
    # mid call/put si existen columnas
    for side in ('call', 'put'):
        b, a = f'{side}_bid', f'{side}_ask'
        if b in df.columns and a in df.columns:
            df[f'{side}_mid'] = (df[b] + df[a]) / 2.0
    if 'strike' in df.columns:
        atm_idx = (df['strike'] - spot).abs().idxmin()
        print('Fila ATM:')
        display(df.loc[[atm_idx]])
    else:
        print('Sin columna strike; columnas=', list(df.columns))
else:
    print('Sin datos de cadena — saltear esta celda.')


Fila ATM:


,symbol,expiration,type,contractSymbol,strike,currency,lastPrice,change,percentChange,volume,openInterest,bid,ask,contractSize,lastTradeDate,impliedVolatility,inTheMoney,Spot,Ticker
21,AAPL,2026-09-21,call,AAPL260921C00335000,335.0,USD,2.5,-1.75,-41.17647,21427.0,2190.0,2.27,2.56,REGULAR,2026-09-18 19:59:54,0.160043,True,336.130005,AAPL


## 5) BYMA / scrapers

Los scrapers (IOL, etc.) son útiles pero **frágiles** en clase. Quedan para el Eje 4 opcional o demos del docente.

Preguntas Hull / lectura: ¿qué información mínima necesitás para armar un bull call spread con datos de mercado?


## Resumen

- Usá `Codigo.data.market_data` (igual que la app).
- Flujo: spot → expiries → chain → ATM/mid.
- Si falla la red: mensaje claro, no pantallazo rojo.
- **Siguiente:** Eje 4a/4b profundizan el panel; la página *Market Data* es el laboratorio interactivo.
